In [0]:
# ==================================================
# GOLD — Evolução Temporal
# Pipeline PRF Acidentes de Trânsito
# ==================================================

from pyspark.sql.functions import col, sum, count, round

# Lê da Silver
df = spark.table("workspace.default.silver_acidentes")

# Agrega por mês
df_gold_temporal = (df
    .groupBy("ano", "mes", "dia_semana")
    .agg(
        count("id").alias("total_acidentes"),
        sum("mortos").alias("total_mortos"),
        sum("feridos").alias("total_feridos"),
        sum("feridos_graves").alias("total_feridos_graves"),
        sum("feridos_leves").alias("total_feridos_leves")
    )
    .withColumn("letalidade",
        round(col("total_mortos") / col("total_acidentes"), 4))
    .orderBy("ano", "mes", "dia_semana")
)

# Grava Gold
(df_gold_temporal.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_evolucao_temporal")
)

print("Gold evolução temporal gravada!")
df_gold_temporal.show(15)